<a href="https://colab.research.google.com/github/ivanadelgado/Yimi-Reynaga-/blob/main/Yimy_Reynaga_QCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qualitative Comparative Analysis (fsQCA)

Este notebook comienza **después de la calibración** realizada en `preQCA`.

**Entrada:** `qca_calibrada.csv`

## ¿Qué haremos y para qué?

1. **Leer la base calibrada** para verificar la representación de los casos que entra al análisis.
2. **Evaluar necesidad** para preguntar si alguna condición debe estar presente cuando aparece `EXITO`.
3. **Construir la tabla de verdad** para organizar las combinaciones de condiciones observadas.
4. **Identificar configuraciones suficientes** para preguntar qué combinaciones se asocian sistemáticamente con `EXITO`.
5. **Simplificar las configuraciones** mediante minimización lógica para obtener una explicación más parsimoniosa.
6. **Evaluar la solución** mediante consistencia, cobertura y PRI.

La guía explica primero qué busca cada paso. Luego ejecutamos un bloque pequeño, observamos su resultado y regresamos a la guía para interpretarlo.

Usaremos `setqca`, una implementación de fsQCA para Python.


## Instalar la librería

En Google Colab necesitamos instalar `setqca` una vez por sesión.


In [ ]:
!pip -q install setqca


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 3.0 MB/s eta 0:00:00


## QCA-01: leer la base preparada

Cada fila representa una reforma constitucional.

`FORTALEZA`, `APOYO`, `MAYORIA` y `EXITO` son membresías fuzzy entre 0 y 1.


In [ ]:
import pandas as pd

qca = pd.read_csv("https://github.com/ivanadelgado/Yimi-Reynaga-/raw/refs/heads/main/qca_calibrada%20(1).csv$0")
qca.head()


,pais,LEGITIMIDAD,MAGNITUD,ELECTORES,DURACION
0,Albania,0.187419,0.071425,7.725983e-01,0.500000
1,Andorra,0.582660,0.038712,9.988146e-01,0.500000
2,Argentina,0.312820,0.067279,6.748073e-10,0.500000
3,Australia,0.402235,0.036297,9.913014e-01,0.813395
4,Austria,0.843980,0.038316,3.063125e-01,0.050000


## QCA-02: buscar condiciones necesarias

Una condición necesaria responde a la pregunta:

> Cuando encontramos el resultado, ¿encontramos también la condición?

Usaremos `necessity_analysis()` para evaluar las tres condiciones respecto de
`LEGITIMIDAD`.

Tomamos 0.90 como referencia para identificar una relación de necesidad fuerte.


In [ ]:
from setqca import necessity_analysis

necesidad = necessity_analysis(
    qca,
    outcome="LEGITIMIDAD",
    conditions=["MAGNITUD", "ELECTORES", "DURACION"],
    consistency_threshold=0.90
)

necesidad.to_frame()[
    ["condition", "consistency", "coverage", "necessary"]
]


,condition,consistency,coverage,necessary
0,~MAGNITUD,0.929262,0.528704,False
1,~DURACION,0.845261,0.683035,False
2,ELECTORES,0.645843,0.570057,False
3,DURACION,0.555643,0.741747,False
4,~ELECTORES,0.468799,0.549160,False
5,MAGNITUD,0.156302,0.682581,False


### ¿Cómo leer el resultado?

La columna principal para nuestra pregunta es `consistency`.

Una condición cercana a 1 estaría presente de manera sistemática cuando aparece
`LEGITIMIDAD`.

En este ejercicio esperamos observar que ninguna condición individual alcanza
el umbral de 0.90. Eso nos lleva a buscar **combinaciones**.



**INTERPRETACIÓN**
Ninguna de las condiciones evaluadas, ni su ausencia, resulta necesaria para LEGITMIDAD bajo
el criterio utilizado en este ejercicio.

## QCA-03: observar combinaciones

Con tres condiciones existen `2³ = 8` configuraciones lógicamente posibles.

`build_truth_table()` organiza esas configuraciones. Los casos se ubican en una
fila según estén más dentro o más fuera de cada conjunto respecto del punto
0.5.

La consistencia de cada fila, sin embargo, se calcula utilizando las membresías
fuzzy originales.


In [ ]:
from setqca import build_truth_table

tabla = build_truth_table(
    qca,
    outcome="LEGITIMIDAD",
    conditions=["MAGNITUD", "ELECTORES", "DURACION"],
    inclusion_cutoff=0.80,
    pri_cutoff=0.75,
    frequency_cutoff=1,
    case_id="caso"
)

tabla_df = tabla.to_frame()

tabla_df


,MAGNITUD,ELECTORES,DURACION,minterm,n,consistency,PRI,OUT,cases,excluded_because
0,0,0,0,0,9,0.762552,0.572323,C,"Austria, Armenia, India, Indonesia, Italy, Pak...",PRI 0.572 below the cutoff of 0.75
1,0,0,1,1,17,0.714655,0.291329,C,"Argentina, Bosnia Herzegovina, Brazil, Chile, ...",PRI 0.291 below the cutoff of 0.75
2,0,1,0,2,14,0.692838,0.441272,0,"Bangladesh, Bolivia, Myanmar, Cyprus, France, ...",consistency 0.693 below the inclusion cutoff o...
3,0,1,1,3,29,0.808183,0.580604,C,"Albania, Andorra, Australia, Bulgaria, Belarus...",PRI 0.581 below the cutoff of 0.75
4,1,0,0,4,1,0.882089,0.426967,C,Kyrgyzstan,PRI 0.427 below the cutoff of 0.75
5,1,0,1,5,4,0.935774,0.506686,C,"Montenegro, Netherlands, Serbia, Slovakia",PRI 0.507 below the cutoff of 0.75
6,1,1,0,6,0,1.000000,1.000000,R,,frequency 0 below the cutoff of 1
7,1,1,1,7,0,1.000000,1.000000,R,,frequency 0 below the cutoff of 1


## QCA-04: identificar configuraciones suficientes

Ahora preguntamos:

> Cuando aparece una configuración, ¿aparece sistemáticamente el resultado?

En nuestra tabla, `OUT = 1` identifica las filas que alcanzan el criterio de
suficiencia definido en 0.80.

Mostramos solamente las columnas necesarias para interpretar esa decisión.


In [ ]:
columnas = [
    "MAGNITUD", "ELECTORES", "DURACION",
    "n", "consistency", "OUT"
]

suficientes = tabla_df[
    tabla_df["OUT"].astype(str) == "1"
][columnas]

suficientes


,MAGNITUD,ELECTORES,DURACION,n,consistency,OUT


### ¿Qué observamos?

La tabla filtrada permite identificar qué configuraciones alcanzan el criterio de suficiencia.

Con solo tres condiciones sería posible inspeccionar estas filas a simple vista, pero esa
inspección manual deja de ser práctica apenas crece el número de condiciones. Por eso no la
hacemos aquí: en QCA-05 usamos minimización lógica para que Python encuentre la
simplificación de forma sistemática.


**INTERPRETACIÓN**
Ninguna combinación de las variables es necesaria, las 3 primeras combinaciones, son excluidas por no alcanzar el parámetro de consisttencia, las otras 3 el del PRI y las 2 restantes no presentasn casos.

**DE AQUÍ A ABAJO YA NO SE REALIZA EL ANÁLISIS PORQUE NO SE ENCONTRARON COMBINACIONES NECESARIAS **

## QCA-05: simplificar las configuraciones

Ahora utilizamos `FSQCA`.

El objeto realiza la minimización lógica sobre la tabla de verdad y conserva las
membresías fuzzy para evaluar la solución.

Usamos:

- consistencia mínima = 0.80;
- frecuencia mínima = 1;
- PRI mínimo = 0.75, el mismo criterio que ya aplicamos en la tabla de verdad de QCA-03.


In [ ]:
from setqca import FSQCA

modelo_qca = FSQCA(
    consistency=0.80,
    pri=0.75,
    frequency=1
)

resultado_qca = modelo_qca.fit(
    qca,
    outcome="EXITO",
    conditions=["FORTALEZA", "APOYO", "MAYORIA"],
    case_id="caso"
)


### ¿Qué pasó?

Si el bloque se ejecutó sin errores, el análisis QCA fue realizado y quedó
almacenado en `resultado_qca`.

En el siguiente bloque extraemos la solución que necesitamos interpretar.


In [ ]:
resultado_qca.summary_frame("conservative")


### ¿Cómo leer el resultado?

La salida muestra la solución conservadora obtenida por la minimización.

La guía interpreta aquí los símbolos `+` y `*` —lo necesario para poder leer la fórmula—.
La interpretación completa (cuántas rutas hay, qué significan, y la equifinalidad) se hace en
QCA-06, junto con las métricas de la solución.


## QCA-06: evaluar la solución

Finalmente extraemos los parámetros de ajuste de la solución.

La **consistencia** pregunta qué tan bien se comporta la configuración como
suficiente para el resultado.

La **cobertura** pregunta cuánto del resultado está cubierto por esa
configuración.

El **PRI** pregunta si esa relación de suficiencia podría ser, en realidad,
ambigua entre el resultado y su ausencia.


In [ ]:
resumen = resultado_qca.summary_frame("conservative")

resumen
